### This notebook contains scripts used for taxonomic profiling and functional inference of rat gut microbiome data.

In [ ]:
import os
import glob
import gc

# Enable and run garbage collection to free memory before processing
gc.enable()
gc.collect()

In [ ]:
FILES = glob.glob("../VZK_data/*/")  # Path to directories containing R1.fastq.gz and R2.fastq.gz
# Check path
FILES[0] # e.g. '../VZK_data/int7/'

In [ ]:
## Specify paths to HUMAnN data
# Note: ChocoPhlAn is not used in this analysis, but it is still required for HUMAnN integrity
# e.g. should output HUMAnN configuration file updated: ...
!humann_config --update database_folders nucleotide /databases/chocophlan
!humann_config --update database_folders protein /databases/uniref
!humann_config --update database_folders utility_mapping /databases/utility_mapping

In [ ]:
%%capture cap


for FILE in FILES:

    _, _, ID, _ = FILE.split("/")


    print(f"working with {ID}")
    
    R1 = glob.glob(f"{FILE}**1.fq.gz")[0]
    R2 = glob.glob(f"{FILE}**2.fq.gz")[0]

    R1C = R1.replace(".fq.gz", "_clean.fq.gz")
    R2C = R2.replace(".fq.gz", "_clean.fq.gz")

    R12F = f"{FILE}classified_#c.fq"

    REPORT = f"{FILE}{ID}_fastp_report.html"
    
    ## Assess sequencing quality and perform initial read filtering
    print(f"fastp ...")
    !fastp -i $R1 -I $R2 -o $R1C -O $R2C -h $REPORT -w 16 -q 30 -l 35 -u 40
    print(f"complete\n")
    
    ## Remove potential host-derived and other contaminated reads
    print(f"hostile ...")
    
    # Align against the rat reference genome (GRCr8)
    !hostile clean --fastq1 $R1C --fastq2 $R2C  -o $FILE -t 12

    !rm  $R1C
    !rm  $R2C

    R1CD = R1C.replace('.fq.gz', ".clean_1.fastq.gz")
    R2CD = R2C.replace('.fq.gz', ".clean_2.fastq.gz")

    !hostile clean --fastq1 $R1CD --fastq2 $R2CD  -o $FILE -t 12 --index rattus_norvegicus_db/rattus_norvegicus

    !rm  $R1CD
    !rm  $R2CD

    R1CD = f"{FILE}{ID}_1_clean.clean_1.clean_1.fastq.gz"
    R2CD = f"{FILE}{ID}_2_clean.clean_2.clean_2.fastq.gz"

    print(f"complete\n")
    
    
    ## If necessary, sort paired-end reads to ensure proper read ordering

    # !bbmap/repair.sh in1=$R1CD in2=$R2CD out1=tmp_R1.fastq out2=tmp_R2.fastq outs=singletons.fastq
    
    # !mv tmp_R1.fastq $R1CD
    # !mv tmp_R2.fastq $R2CD
    
    
    ## Perform taxonomic profiling using Kraken2 against the Standard-16 database
    REPORT = f"{FILE}{ID}_report_k2_standard16_v2.txt"
    print(f"kraken2 ...")
    !kraken2 --db ../kraken2_data/k2_standard16 --memory-mapping --confidence 0.05 --minimum-hit-groups 3 --threads 16 --paired --report $REPORT --report-minimizer-data --classified-out $R12F $R1CD $R2CD > /dev/null
    print(f"complete\n")


    R1F = glob.glob(f"{FILE}classified__1c.fq")[0]
    R2F = glob.glob(f"{FILE}classified__2c.fq")[0]
    
    
    fun_path = f"{FILE}/humann/"
    %mkdir -p $fun_path
    
    ## Perform functional inference using HUMAnN against the EC-filtered UniRef90 database
    print(f"humann ...")
    !humann --input $R1F --output $fun_path --protein-database ../humann_light/databases/uniref --bypass-nucleotide-index --bypass-nucleotide-search --bypass-prescreen --translated-identity-threshold 30 --translated-query-coverage-threshold 80 --pathways metacyc --remove-stratified-output --threads 12 --memory-use minimum --diamond-options "-b 0.6 -c 4 -k 5" --remove-temp-output --output-format tsv
    print(f"complete\n\n")

    ## If needed, remove temporary files to free up disk space
    !rm  $R1CD
    !rm  $R2CD
    !rm  $R1F
    !rm  $R2F

    gc.collect()

In [ ]:
# Save processing log for downstream analysis of contamination and classification rates
with open("log.txt", 'w') as f:
    f.write(cap.stdout)  

In [ ]:
# Save Kraken2 database k-mer statistics, including the total number of unique k-mers in each reference genome
!kraken2-inspect --db ../kraken2_data/k2_standard16  > k2_standard16_standard_summary.txt

In [ ]:
# From this cell to the end of the script: create the results directory and copy profiling outputs
%mkdir -p VZK_res
%mkdir -p fastp_report

In [ ]:
F1 = glob.glob("../VZK_data/*/*report_k2_standard16_v2.txt")
F2 = glob.glob("../VZK_data/*/humann/*pathabundance.tsv")

assert len(F1) == len(F2)

In [ ]:
for file in glob.glob("../VZK_data/*/*_fastp_report.html"):
    file2 = file.split("/")[-1]

    !cp $file fastp_report/$file2

In [ ]:
for f1, f2 in zip(F1, F2):
    
    _, _, ID1, ff1 = f1.split('/')
    _, _, ID2, _, ff2 = f2.split('/')
    
    %mkdir -p VZK_res/$ID1
    %mkdir -p VZK_res/$ID2
    
    !cp $f1 VZK_res/$ID1/$ff1
    !cp $f2 VZK_res/$ID2/$ff2